# Prediction Request ke Heart Disease Model Serving

Notebook ini digunakan untuk menguji dan melakukan prediction request ke sistem machine learning yang telah dijalankan di cloud.

**Model:** Heart Disease Classification  
**Serving URL:** Sesuaikan dengan URL deployment Anda  
**Dataset:** Heart Disease UCI  

## 1. Import Library

In [ ]:
import json
import requests
import numpy as np
import pandas as pd
import tensorflow as tf
import base64
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

print('TensorFlow version:', tf.__version__)
print('Semua library berhasil diimport!')

## 2. Konfigurasi URL Serving

Sesuaikan `SERVING_URL` dengan URL deployment yang digunakan.

In [ ]:
# ── Konfigurasi ──────────────────────────────────────────────────────
# Ganti URL ini dengan URL deployment cloud Anda
SERVING_URL = 'https://josa-pratama-heart-disease.railway.app'  # Railway

# Atau gunakan localhost jika berjalan lokal
# SERVING_URL = 'http://localhost:5000'

PREDICT_URL = f'{SERVING_URL}/predict'
HEALTH_URL  = f'{SERVING_URL}/'

print(f'Predict endpoint : {PREDICT_URL}')
print(f'Health endpoint  : {HEALTH_URL}')

## 3. Health Check

In [ ]:
try:
    resp = requests.get(HEALTH_URL, timeout=10)
    resp.raise_for_status()
    print('Status:', resp.status_code)
    print(json.dumps(resp.json(), indent=2))
except requests.exceptions.ConnectionError:
    print('⚠️  Tidak dapat terhubung ke serving URL.')
    print('Pastikan model serving sudah berjalan.')

## 4. Mendefinisikan Helper Function

In [ ]:
def predict_heart_disease(patient_data: dict) -> dict:
    """
    Melakukan prediksi penyakit jantung untuk satu pasien.
    
    Args:
        patient_data: Dictionary berisi fitur pasien.
        
    Returns:
        Dictionary berisi hasil prediksi.
    """
    try:
        response = requests.post(
            PREDICT_URL,
            json=patient_data,
            headers={'Content-Type': 'application/json'},
            timeout=15
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        return {'error': f'HTTP Error: {e}', 'status_code': response.status_code}
    except requests.exceptions.ConnectionError:
        return {'error': 'Connection error – pastikan server berjalan'}
    except Exception as e:
        return {'error': str(e)}


def batch_predict(patients_df: pd.DataFrame) -> pd.DataFrame:
    """
    Melakukan prediksi batch untuk banyak pasien.
    
    Args:
        patients_df: DataFrame berisi data pasien.
        
    Returns:
        DataFrame dengan kolom prediksi tambahan.
    """
    results = []
    for _, row in patients_df.iterrows():
        patient = row.to_dict()
        # Hapus kolom target jika ada
        patient.pop('target', None)
        result = predict_heart_disease(patient)
        results.append(result)
    
    result_df = patients_df.copy()
    result_df['predicted_label']      = [r.get('prediction', None) for r in results]
    result_df['confidence']           = [r.get('confidence', None) for r in results]
    result_df['diagnosis']            = [r.get('diagnosis', None) for r in results]
    result_df['processing_time_ms']   = [r.get('processing_time_ms', None) for r in results]
    return result_df

print('Helper functions siap digunakan!')

## 5. Single Prediction – Contoh Pasien

In [ ]:
# Contoh pasien dengan penyakit jantung (target=1)
patient_positive = {
    'age': 63,
    'sex': 1,
    'cp': 3,
    'trestbps': 145,
    'chol': 233,
    'fbs': 1,
    'restecg': 0,
    'thalach': 150,
    'exang': 0,
    'oldpeak': 2.3,
    'slope': 0,
    'ca': 0,
    'thal': 1
}

print('Data Pasien:')
print(json.dumps(patient_positive, indent=2))
print()

result = predict_heart_disease(patient_positive)
print('Hasil Prediksi:')
print(json.dumps(result, indent=2))

In [ ]:
# Contoh pasien tanpa penyakit jantung (target=0)
patient_negative = {
    'age': 67,
    'sex': 1,
    'cp': 0,
    'trestbps': 160,
    'chol': 286,
    'fbs': 0,
    'restecg': 0,
    'thalach': 108,
    'exang': 1,
    'oldpeak': 1.5,
    'slope': 1,
    'ca': 3,
    'thal': 2
}

print('Data Pasien:')
print(json.dumps(patient_negative, indent=2))
print()

result = predict_heart_disease(patient_negative)
print('Hasil Prediksi:')
print(json.dumps(result, indent=2))

## 6. Batch Prediction dari Dataset

In [ ]:
# Muat dataset
df = pd.read_csv('josa_pratama-pipeline/data/raw/heart.csv')
print(f'Shape dataset: {df.shape}')
display(df.head())

# Ambil 10 sampel untuk testing
test_samples = df.sample(10, random_state=42).reset_index(drop=True)
print(f'\nMelakukan prediksi untuk {len(test_samples)} sampel...')

In [ ]:
# Batch prediction
results_df = batch_predict(test_samples)

# Tampilkan hasil
display_cols = ['age', 'sex', 'target', 'predicted_label', 'confidence', 'diagnosis']
display(results_df[display_cols])

# Hitung akurasi pada sampel ini
correct = (results_df['target'] == results_df['predicted_label']).sum()
total   = len(results_df)
print(f'\nAkurasi pada sampel: {correct}/{total} = {correct/total:.1%}')

## 7. Visualisasi Hasil Prediksi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribusi confidence score
axes[0].hist(
    results_df[results_df['target'] == 1]['confidence'].dropna(),
    bins=10, alpha=0.7, label='Actual Positive', color='red'
)
axes[0].hist(
    results_df[results_df['target'] == 0]['confidence'].dropna(),
    bins=10, alpha=0.7, label='Actual Negative', color='green'
)
axes[0].set_title('Distribusi Confidence Score')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].axvline(x=0.5, color='black', linestyle='--', label='Threshold=0.5')

# Plot 2: Confusion matrix sederhana
if 'predicted_label' in results_df.columns and results_df['predicted_label'].notna().all():
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(
        results_df['target'],
        results_df['predicted_label'].astype(int)
    )
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
        xticklabels=['No Disease', 'Disease'],
        yticklabels=['No Disease', 'Disease']
    )
    axes[1].set_title('Confusion Matrix (Sample)')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('prediction_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot disimpan sebagai prediction_results.png')

## 8. Performance Benchmark

In [ ]:
import time

# Ukur latency untuk 20 request
n_requests = 20
latencies = []

for i in range(n_requests):
    start = time.time()
    result = predict_heart_disease(patient_positive)
    end = time.time()
    if 'error' not in result:
        latencies.append((end - start) * 1000)  # dalam ms

if latencies:
    print(f'📊 Benchmark Results ({len(latencies)} requests):')
    print(f'   Min latency  : {min(latencies):.1f} ms')
    print(f'   Max latency  : {max(latencies):.1f} ms')
    print(f'   Avg latency  : {np.mean(latencies):.1f} ms')
    print(f'   P95 latency  : {np.percentile(latencies, 95):.1f} ms')
    print(f'   Throughput   : {1000/np.mean(latencies):.1f} req/sec')
else:
    print('Tidak ada data latency (pastikan server berjalan)')

## 9. Cek Prometheus Metrics

In [ ]:
metrics_url = f'{SERVING_URL}/metrics'

try:
    resp = requests.get(metrics_url, timeout=5)
    # Tampilkan beberapa metric yang relevan
    for line in resp.text.split('\n'):
        if 'heart_disease' in line and not line.startswith('#'):
            print(line)
except Exception as e:
    print(f'Tidak dapat mengambil metrics: {e}')